RAG

chunck -> a small piece of a document

embedding -> A vector (list of numbers) representing text meaning

vector database -> a database that scores and searches vectors by similarity

retrieval -> finding the most relevant chunks for a given query

context injection -> Adding retrive chunks into the llm's prompt

grounding -> forcing the llm to answer based on the provided facts

knowledge base -> the collection of documents the system can retrieve from

In [ ]:
#xxxxxxxxxxxxxx

In [ ]:
#RAG

In [ ]:
# ! runs a terminal command from inside the
#this is how the install packaes in google colab
#sentence transfoermaers :

In [ ]:
!pip install sentence-transformers chromadb groq pandas -q

In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
print("All the libraries imported succesfully")
print("Ready to build a RAG system")

All the libraries imported succesfully
Ready to build a RAG system


In [ ]:
import os
groq_api_key = "xxxxxx"
os.environ["GROQ_API_KEY"] = groq_api_key
groq_client =Groq(api_key=groq_api_key)
print("groq api client initiated")
print("Note : If you see an authenticated error later , double check you api key")

groq api client initiated
Note : If you see an authenticated error later , double check you api key


In [ ]:
df=pd.read_csv('college_notes.csv')
print("Shape of the dataset: ",df.shape)
print("\nColumn names: ",df.columns.tolist())

Shape of the dataset:  (15, 4)

Column names:  ['note_id', 'subject', 'topic', 'content']


In [ ]:
print("print the first 3 rows")
print(df.head(3))

print the first 3 rows
  note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  


In [ ]:
print("Subject in the dataset")
print(df['subject'].value_counts)

Subject in the dataset
<bound method IndexOpsMixin.value_counts of 0       Data Engineering
1       Data Engineering
2       Data Engineering
3       Data Engineering
4       Data Engineering
5       Machine Learning
6       Machine Learning
7       Machine Learning
8       Machine Learning
9       Machine Learning
10         Generative AI
11         Generative AI
12         Generative AI
13    Python Programming
14    Python Programming
Name: subject, dtype: object>


In [ ]:
print("Sample of topics")
print(df[['note_id', 'subject','topic']].to_string(index=False))

Sample of topics
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Python Programming                 Pandas Library
   N015 Python Programming             Data Visualization


In [ ]:
print("Length of content ( number of characters) for each note:")
df['content_length']=df['content'].apply(len)
print(df[['topic', 'content_length']].to_string(index=False))

Length of content ( number of characters) for each note:
                         topic  content_length
                 ETL Pipelines             216
                 SQL Databases             209
                 Data Cleaning             210
      APIs and Data Collection             224
          Big Data and PySpark             242
           Supervised Learning             255
              Model Evaluation             240
           Feature Engineering             236
                Decision Trees             227
                 Random Forest             238
         Large Language Models             226
            Prompt Engineering             275
Retrieval Augmented Generation             274
                Pandas Library             252
            Data Visualization             249


In [ ]:
#chunking -> spliting the large documents into small chunks to retrieve easily
#Ex: ask question from text book vs paragraph
documents=df['content'].tolist()
#convert to list and store in the variable
ids=[f"note_{row['note_id']}" for row in df.to_dict('records')]
#convert into dictionary : key and values
metadatas = [
    {"subject": row['subject'], "topic": row['topic']}
    for row in df.to_dict('records')
]
print(f"Total chunks prepared : {len(documents)}")
print(f"First document ID : {ids[0]}")
print(f"First document metadata : {metadatas[0]}")
print(f"First document content : {documents[0][:100]}")


Total chunks prepared : 15
First document ID : note_N001
First document metadata : {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First document content : ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc


In [ ]:
#embedding

In [ ]:
print("Loading embedding model...")
print("(subsequent runs will be faster as the model is cached)")
embedding_model=SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded")

Loading embedding model...
(subsequent runs will be faster as the model is cached)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded


In [ ]:
#Quick test : embed the one sentence and check the vector shape
#convert text into list of numbers
test_embedding=embedding_model.encode("This is a test sentence.")
print(f"Test embedding shape: {test_embedding.shape}")


#First five values of test embedding
print(f"First five values of test embedding: {test_embedding[:5]}")


Test embedding shape: (384,)
First five values of test embedding: [0.08429647 0.05795366 0.00449333 0.1058211  0.00708344]


In [ ]:
#create the client and connection
chroma_client=chromadb.Client()

#collection of the db
collection=chroma_client.get_or_create_collection(name="college_notes_rag")
print("chromadb cluent created")
print(f"Collection name : college_notes_reg")

#no of collection in the chromadb
print(f"Documents in the collection so far: {collection.count()}")

chromadb cluent created
Collection name : college_notes_reg
Documents in the collection so far: 0


In [ ]:
#embedding the text into list of numbers
embeddings=embedding_model.encode(documents,show_progress_bar=True)
print(f"\nEmbedding matrix shape: {embeddings.shape}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape: (15, 384)


In [ ]:
#convert the numpy array to python lists
embeddings_list=embeddings.tolist()

#Add into the collection
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
    embeddings=embeddings_list
)

print(f"Document successsfully added into the chromadb")
print(f"Total documents in the collection : {collection.count()}")

Document successsfully added into the chromadb
Total documents in the collection : 15


In [ ]:
def retrieve_relevant_chunks(question, top_k=3):
  #retrieve the top 3 relatd chunks related to given question
  question_embedding=embedding_model.encode(question)


  #encode the question
  #Query from the database (chroma)
  results=collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k
  )
  return results;
  print("retrieve function defined successfully")

In [ ]:
#test the function with question
test_question ="What is ETL and how does it work in data engineering?"
print(f"Question : {test_question}")
print()
results=retrieve_relevant_chunks(test_question,top_k=3)
for i,(doc, dist, meta, doc_id) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0],
    results['ids'][0])):
  print(f"\nResult {i+1}:")
  print(f"Document ID : {doc_id}")
  print(f"Subject : {meta['subject']}")
  print(f"Topic : {meta['topic']}")
  print(f"Distance : {dist:.4f}")
  print(f"Content : {doc[:120]}...")

Question : What is ETL and how does it work in data engineering?


Result 1:
Document ID : note_N001
Subject : Data Engineering
Topic : ETL Pipelines
Distance : 0.2269
Content : ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i...

Result 2:
Document ID : note_N004
Subject : Data Engineering
Topic : APIs and Data Collection
Distance : 1.0690
Content : An API or Application Programming Interface allows two software applications to talk to each other. In data engineering ...

Result 3:
Document ID : note_N015
Subject : Python Programming
Topic : Data Visualization
Distance : 1.3375
Content : Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo...


In [ ]:
#context injection -
#connecting into llm prompt which ever retrieve  from the RAG model (document search)

In [ ]:
#rag prompt template

system:
"prompt"

---
user:

context:

[retrieve doc1]

[retrieve doc2]

[retrieve doc3]

---

Question : [user's Question]

Answer :

In [ ]:
#create the function to connect and print the result where the llm prompt asked
def build_context_from_results(results):
  #result of collection.query()
  context_parts=[] #to collect the formated chuncks

  #loop to retrieve the document and metadata
  for i, (doc,meta) in enumerate(zip(
      results['documents'][0], #text data
      results['metadatas'][0])):

      #formate the chunck data into this
      chunk_text=f"[Source {i+1} : {meta['subject']} - {meta['topic']}]\n{doc}"
      #append the chunck into created context_parts
      context_parts.append(chunk_text)

  #make the context_parts into str and return str in the function
  context_str="\n\n---\n\n".join(context_parts)
  return context_str

In [ ]:
#Test the above function and print the result
context = build_context_from_results(results)
print("Build context string from retrieved chuncks:")
print()
print(context[:500]+"...")
print(f"\nTotal length of the context : {len(context)} characters")

Build context string from retrieved chuncks:

[Source 1 : Data Engineering - ETL Pipelines]
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.

---

[Source 2 : Data Engineering - APIs and Data Collection]
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services lik...

Total length of the context : 859 characters


In [ ]:
#build the RAG generation function
#This function sends the context +question to groq llm and returns the answer

#there is two prompt : system and user
def generate_rag_answer(question,context):
  system_prompt="""
  You are a helpful academic assistent for engineering students.
  You will be given context retrieved from a college knowldge base, and a student question.
  RULES:
  1.Answer only using the information provided in the context below.
  2.If the answer is not found in the context, say exactly:
  "I don't have the enough knowledge base answer to answer this question."
  3.Do not use your general training knowledge.
  4.keep answers clear, accurate, and beginner friendly.
  5.mention which source the information came from when possible."""

  #user prompt
  user_prompt = f"""context from knowledge Base:
{context}
---
Student's question : {question}
please answer the question using the context provided above."""


#call the groq api with the msg
  response=groq_client.chat.completions.create(
      #model : llm
      model="llama-3.1-8b-instant",
      #msg for groq
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt}
      ],
      #allocate the temperature value and maximum token value
      temperature=0.1,
      max_tokens=500
  )
  #Extract the answer from api response object
  #response.choices : a list of response options (usually 1)
  #[0] : take the first ( and only ) choice
  #.message.content : the actual text of the response
  answer = response.choices[0].message.content
  return answer
print("RAG generation function defined")



RAG generation function defined


In [ ]:
# CELL 14

# Complete RAG Pipeline:
# Question -> Retrieve Relevant Documents -> Build Context -> Generate Answer

def ask_college_assistant(question, top_k=3, verbose=True):

    if verbose:
        print(f"Question: {question}")
        print()

        # Step 1: Retrieve relevant chunks from the vector database
        print("Step 1: Retrieving relevant documents...")

    results = retrieve_relevant_chunks(question, top_k=top_k)

    if verbose:
        print(f"Retrieved {top_k} chunks from the knowledge base:")

        for i, meta in enumerate(results['metadatas'][0]):
            print(f"{i+1}. {meta['subject']} - {meta['topic']}")

        print("\nStep 2: Building context string...")

    # Step 2: Inject retrieved chunks into a single context string
    context = build_context_from_results(results)

    if verbose:
        print(f"Context built ({len(context)} characters)")
        print("\nStep 3: Sending context and question to the LLM for answer generation")

    # Step 3: Generate answer using the LLM
    answer = generate_rag_answer(question, context)

    if verbose:
        print()
        print("Answer:")
        print(answer)
        print()

    return answer


print("Complete RAG pipeline function is ready!")
print("Use: ask_college_assistant(question, top_k=3)")

Complete RAG pipeline function is ready!
Use: ask_college_assistant(question, top_k=3)


In [ ]:
#Test the above function
question_1 = 'What is ETL and what are its three main stages?'
answer1=ask_college_assistant(question_1, top_k=3, verbose=True)

Question: What is ETL and what are its three main stages?

Step 1: Retrieving relevant documents...
Retrieved 3 chunks from the knowledge base:
1. Data Engineering - ETL Pipelines
2. Generative AI - Retrieval Augmented Generation
3. Generative AI - Prompt Engineering

Step 2: Building context string...
Context built (933 characters)

Step 3: Sending context and question to the LLM for answer generation

Answer:
Based on the context provided, I can answer the student's question.

ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources (Extract), transforming it into a clean and structured format (Transform), and loading it into a database or data warehouse for analysis (Load).

The three main stages of ETL are:

1. **Extract**: Collecting raw data from different sources.
2. **Transform**: Transforming the raw data into a clean and structured format.
3. **Load**: Loading the transformed data into a database or data warehouse for analysis.

T

In [ ]:
question_2="How do embeddings help in building search systems?"
answer2=ask_college_assistant(question_2, top_k=3, verbose=True)

Question: How do embeddings help in building search systems?

Step 1: Retrieving relevant documents...
Retrieved 3 chunks from the knowledge base:
1. Generative AI - Retrieval Augmented Generation
2. Generative AI - Large Language Models
3. Machine Learning - Feature Engineering

Step 2: Building context string...
Context built (913 characters)

Step 3: Sending context and question to the LLM for answer generation

Answer:
Unfortunately, the context provided does not directly address how embeddings help in building search systems. However, I can try to provide a related answer based on the information given.

From [Source 1: Generative AI - Retrieval Augmented Generation], we know that RAG (Retrieval Augmented Generation) is a technique where an AI model first retrieves relevant documents from a knowledge base and then generates an answer based on those retrieved documents. This process involves matching the query with the knowledge base to retrieve relevant documents.

Embeddings can 

In [ ]:
question3 = "What is the population of Tokyo?"
print("Testing with an out-of-scope question (not in college notes):")
answer3=ask_college_assistant(question3,top_k=3,verbose=True)

Testing with an out-of-scope question (not in college notes):
Question: What is the population of Tokyo?

Step 1: Retrieving relevant documents...
Retrieved 3 chunks from the knowledge base:
1. Generative AI - Large Language Models
2. Data Engineering - SQL Databases
3. Data Engineering - Data Cleaning

Step 2: Building context string...
Context built (802 characters)

Step 3: Sending context and question to the LLM for answer generation

Answer:
I don't have the enough knowledge base answer to answer this question.

The context provided does not contain any information about the population of Tokyo. It covers topics such as Generative AI, SQL databases, and data cleaning, but does not include demographic data or information about specific cities.



| Feature | Without RAG | With RAG |
|----------|------------|----------|
| Knowledge Source | LLM Training Data | Your Custom Documents |
| Hallucination Risk | High | Low |
| Can Use Private Data | No | Yes |
| Knowledge Cutoff | Yes (Limited to Training Data) | No (You Can Add New Documents Anytime) |
| Cites Sources | No | Yes |
| Cost | Cheaper | Slightly Higher |

In [ ]:
#practice Exercise function and retrieve like above
def retrieve_by_subject(question,subject_filter,top_k=2):
  #question:user question, subject_filter: topic like genAI alone , top_k: (top results)

  #Embed the question for similarity checking
  question_embedding=embedding_model.encode(question)

  #Query to fetch the data from the chromadb
  results=collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k,
      where={"subject": subject_filter}
      #where : A filter codn applied metadata
  )
  return results




In [ ]:
#Task :Ask  a question to retrieve the genai data
print("Retrieving only from genAI subject:")
#call the function(above)
filtered_results=retrieve_by_subject(
    question="How do LLM generate text?",
    subject_filter="genAI",
    top_k=2
)

#print the result of the above function call
for i,(doc,meta) in enumerate(zip(
    filtered_results['documents'][0],
    filtered_results['metadatas'][0])):
    print(f"\nResult {i+1}: [{meta['subject']}]{meta['topic']}")
    print(f"  Content: {doc[:100]}...")




Retrieving only from genAI subject:



---
Beginner Questions

q1. What is hallucination in the context of LLMs?

q2. What does RAG stand for? What problem does it solve?

q3. What us the role of a vector databse in the RAG pipeline?


Intermediate question

---

q4.What is the difference between the indexing phase and querying phase

q5.Why must you use the same embedding model for both documnets and queries

q6.Why is the low temperature (eg.0.1)preferd for RAG based llm calls?


Coding Question

---
q7.Modify the ask_college_assistent() fun to also display the distances scores of retrieved chunks in the output

q8.Change the system prompt in generate_rag_answer() to instruct the llm to always respond in bullet points.

q9.Add a function that returns only the topic names of retrieved chuncks without their full content

# Beginner Questions

### Q1. What is hallucination in the context of LLMs?

Hallucination occurs when an LLM generates information that appears correct but is actually false, inaccurate, or unsupported by facts.

### Q2. What does RAG stand for? What problem does it solve?

RAG stands for Retrieval-Augmented Generation. It solves the problem of outdated knowledge and hallucinations by retrieving relevant information from external documents before generating an answer.

### Q3. What is the role of a vector database in the RAG pipeline?

A vector database stores document embeddings and performs similarity search to retrieve the most relevant documents for a user's query.

---

# Intermediate Questions

### Q4. What is the difference between the indexing phase and querying phase?

Indexing Phase:
Documents are processed, converted into embeddings, and stored in the vector database.

Querying Phase:
The user's question is converted into an embedding, relevant documents are retrieved, and the LLM generates an answer using the retrieved context.

### Q5. Why must you use the same embedding model for both documents and queries?

Using the same embedding model ensures that documents and queries are represented in the same vector space, allowing accurate similarity comparisons.

### Q6. Why is a low temperature (e.g., 0.1) preferred for RAG-based LLM calls?

A low temperature reduces randomness and helps the model generate more factual, consistent, and context-grounded answers based on the retrieved documents.

---

# Coding Questions

### Q7. Modify the ask_college_assistant() function to also display the distance scores of retrieved chunks in the output.

Retrieve the distance scores from the vector database results and display them along with each retrieved chunk to indicate how closely the chunk matches the user's query.

### Q8. Change the system prompt in generate_rag_answer() to instruct the LLM to always respond in bullet points.

Update the system prompt with an instruction such as:
"Always provide answers in bullet-point format."

### Q9. Add a function that returns only the topic names of retrieved chunks without their full content.

Create a function that retrieves relevant chunks and returns only the topic names from the metadata instead of returning the full document content.

# **MINIPROJECT : COLLEGE KNOWLEDGE ASSISTENT**

Project description

build a complete college knowledge assistent that:

1.Loads the college_notes.csv knowledge base

2.Indexes all notes in chromadb with embedding

3.Accept a student question

4.Retrieves the top 3 relevant notes

5.Injects them as context into groq llm prompt

6.Returns a clear, grounded answer with source citations

7.Handles questions outside the knowledge base gracefully

In [ ]:
# ==========================================================
# MINI PROJECT : COLLEGE KNOWLEDGE ASSISTANT
# ==========================================================

import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq

# ----------------------------------------------------------
# 1. Load Knowledge Base
# ----------------------------------------------------------

df = pd.read_csv("college_notes.csv")

print(f"Loaded {len(df)} notes")

# ----------------------------------------------------------
# 2. Initialize Embedding Model
# ----------------------------------------------------------

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# ----------------------------------------------------------
# 3. Create ChromaDB Collection
# ----------------------------------------------------------

client_db = chromadb.Client()

collection = client_db.get_or_create_collection(
    name="college_notes"
)

# ----------------------------------------------------------
# 4. Store Notes with Embeddings
# ----------------------------------------------------------

documents = []
metadatas = []
ids = []

for i, row in df.iterrows():

    text = str(row["content"])

    documents.append(text)

    metadatas.append({
        "subject": row["subject"],
        "topic": row["topic"]
    })

    ids.append(str(i))

embeddings = embedding_model.encode(documents).tolist()

collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print("Knowledge Base Indexed Successfully!")

# ----------------------------------------------------------
# 5. Initialize Groq
# ----------------------------------------------------------

groq_client = Groq(
    api_key="xxxxx"
)

# ----------------------------------------------------------
# 6. RAG Function
# ----------------------------------------------------------

def ask_college_assistant(question, top_k=3):

    # Retrieve Relevant Notes
    query_embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    # Handle Out-of-Scope Questions
    if len(docs) == 0:
        return "Sorry, this information is not available in the college knowledge base."

    # Build Context
    context = "\n\n".join(docs)

    prompt = f"""
You are a College Knowledge Assistant.

Answer ONLY using the provided context.

Context:
{context}

Question:
{question}

If the answer is not present in the context,
say:
'Sorry, this information is not available in the college knowledge base.'
"""

    # Generate Answer
    response = groq_client.chat.completions.create(
    model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3
    )

    answer = response.choices[0].message.content

    # Add Citations
    citations = "\n\nSources:\n"

    for meta in metas:
        citations += f"- {meta['subject']} : {meta['topic']}\n"

    return answer + citations

# ----------------------------------------------------------
# 7. Ask Question
# ----------------------------------------------------------

question = input("Ask a Question: ")

answer = ask_college_assistant(question)

print("\nAnswer:\n")
print(answer)

Loaded 15 notes


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Knowledge Base Indexed Successfully!
Ask a Question: How is llm work?

Answer:

A Large Language Model (LLM) such as GPT, Claude, or LLaMA works by being trained on massive amounts of text data. It can generate human-like text, answer questions, summarize documents, and perform many language tasks.

Sources:
- Generative AI : Large Language Models
- Generative AI : Retrieval Augmented Generation
- Machine Learning : Random Forest

